# **Leaderboard**

**NetBenefit(k) = 1000 * (ATE_like - k * ATE_dislike)**

k - a penalty const that shows how costly a dislike is relative to a like 

**Impact(k) =  Ns/1000 * NetBenefit(k)** 

Ns - sample size in a segment after trimming

- Rank based om **Impact**
- Make decisions based on **NetBenefit**

#### **Interpretation**

1. NetBenefit > 2 ==> Increase Algo
2. NetBenefit < -2 ==> Decrease Algo
3. NetBenefit ∈ [-2,2] ==> Not significant



In [1]:
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
from dotenv import load_dotenv
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import gc

In [2]:
sys.path.append(os.path.abspath("../src"))
from leaderboard import Leaderboard

In [3]:
load_dotenv()
CSV_PATH = os.getenv("CSV_PATH")
csv_path = CSV_PATH

In [4]:
leaderboard = Leaderboard(csv_path)


**Check different k values**

In [5]:
# k = 0.5
lbd_05 = leaderboard.rank_segments(k=0.5)
lbd_05

,segment,n_trimmed,ATE_like,ATE_dislike,NetBenefit,Impact
0,segment_2,29609335,0.00909,0.00124,8.470,250791.067
1,segment_3,27520720,0.00925,0.00152,8.490,233650.913
2,segment_5,18346619,0.00946,0.00151,8.705,159707.318
3,segment_4,13670051,0.00791,0.00082,7.500,102525.382
4,segment_1,9256718,0.00940,0.00067,9.065,83912.149
5,segment_10,460646,0.07575,0.00410,73.700,33949.610
6,segment_6,905999,0.00922,0.00202,8.210,7438.252
7,segment_8,452769,0.01245,0.00356,10.670,4831.045
8,segment_7,833382,0.00686,0.00306,5.330,4441.926
9,segment_9,431926,0.00772,0.00268,6.380,2755.688


In [6]:
# k = 1
lbd_1 = leaderboard.rank_segments()
lbd_1

,segment,n_trimmed,ATE_like,ATE_dislike,NetBenefit,Impact
0,segment_2,29609335,0.00909,0.00124,7.85,232433.280
1,segment_3,27520720,0.00925,0.00152,7.73,212735.166
2,segment_5,18346619,0.00946,0.00151,7.95,145855.621
3,segment_4,13670051,0.00791,0.00082,7.09,96920.662
4,segment_1,9256718,0.00940,0.00067,8.73,80811.148
5,segment_10,460646,0.07575,0.00410,71.65,33005.286
6,segment_6,905999,0.00922,0.00202,7.20,6523.193
7,segment_8,452769,0.01245,0.00356,8.89,4025.116
8,segment_7,833382,0.00686,0.00306,3.80,3166.852
9,segment_9,431926,0.00772,0.00268,5.04,2176.907


In [7]:
# k = 2
lbd_2 = leaderboard.rank_segments(k=2)
lbd_2

,segment,n_trimmed,ATE_like,ATE_dislike,NetBenefit,Impact
0,segment_2,29609335,0.00909,0.00124,6.61,195717.704
1,segment_3,27520720,0.00925,0.00152,6.21,170903.671
2,segment_5,18346619,0.00946,0.00151,6.44,118152.226
3,segment_4,13670051,0.00791,0.00082,6.27,85711.220
4,segment_1,9256718,0.00940,0.00067,8.06,74609.147
5,segment_10,460646,0.07575,0.00410,67.55,31116.637
6,segment_6,905999,0.00922,0.00202,5.18,4693.075
7,segment_8,452769,0.01245,0.00356,5.33,2413.259
8,segment_9,431926,0.00772,0.00268,2.36,1019.345
9,segment_7,833382,0.00686,0.00306,0.74,616.703


In [8]:
# k = 3
lbd_3 = leaderboard.rank_segments(k=3)
lbd_3

,segment,n_trimmed,ATE_like,ATE_dislike,NetBenefit,Impact
0,segment_2,29609335,0.00909,0.00124,5.37,159002.129
1,segment_3,27520720,0.00925,0.00152,4.69,129072.177
2,segment_5,18346619,0.00946,0.00151,4.93,90448.832
3,segment_4,13670051,0.00791,0.00082,5.45,74501.778
4,segment_1,9256718,0.00940,0.00067,7.39,68407.146
5,segment_10,460646,0.07575,0.00410,63.45,29227.989
6,segment_6,905999,0.00922,0.00202,3.16,2862.957
7,segment_8,452769,0.01245,0.00356,1.77,801.401
8,segment_9,431926,0.00772,0.00268,-0.32,-138.216
9,segment_7,833382,0.00686,0.00306,-2.32,-1933.446


In [9]:
# k = 5
lbd_5 = leaderboard.rank_segments(k=5)
lbd_5

,segment,n_trimmed,ATE_like,ATE_dislike,NetBenefit,Impact
0,segment_2,29609335,0.00909,0.00124,2.89,85570.978
1,segment_1,9256718,0.00940,0.00067,6.05,56003.144
2,segment_4,13670051,0.00791,0.00082,3.81,52082.894
3,segment_3,27520720,0.00925,0.00152,1.65,45409.188
4,segment_5,18346619,0.00946,0.00151,1.91,35042.042
5,segment_10,460646,0.07575,0.00410,55.25,25450.691
6,segment_6,905999,0.00922,0.00202,-0.88,-797.279
7,segment_8,452769,0.01245,0.00356,-5.35,-2422.314
8,segment_9,431926,0.00772,0.00268,-5.68,-2453.340
9,segment_7,833382,0.00686,0.00306,-8.44,-7033.744


**Ranking matrix**

In [ ]:
leaderboards = {
    0.5: lbd_05,
    1: lbd_1,
    2: lbd_2,
    3: lbd_3,
    5: lbd_5
}
leaderboards

rank_tables = []

for k, df in leaderboards.items():
    temp = df.copy()
    temp["rank"] = range(1, len(temp) + 1)
    temp = temp[["segment", "rank"]]
    temp = temp.rename(columns={"rank": f"k={k}"})
    rank_tables.append(temp)

rank_matrix = reduce(
    lambda left, right: pd.merge(left, right, on="segment", how="outer"),
    rank_tables
)

rank_matrix["avg rank"] = rank_matrix[[f"k={k}" for k in leaderboards.keys()]].mean(axis=1)
rank_matrix = rank_matrix.sort_values("avg rank")
rank_matrix["std dev"] = round(rank_matrix[[f"k={k}" for k in leaderboards.keys()]].std(axis=1), 3)
rank_matrix

,segment,k=0.5,k=1,k=2,k=3,k=5,avg rank,std dev
2,segment_2,1,1,1,1,1,1.0,0.000
3,segment_3,2,2,2,2,4,2.4,0.894
5,segment_5,3,3,3,3,5,3.4,0.894
4,segment_4,4,4,4,4,3,3.8,0.447
0,segment_1,5,5,5,5,2,4.4,1.342
1,segment_10,6,6,6,6,6,6.0,0.000
6,segment_6,7,7,7,7,7,7.0,0.000
8,segment_8,8,8,8,8,8,8.0,0.000
9,segment_9,10,10,9,9,9,9.4,0.548
7,segment_7,9,9,10,10,10,9.6,0.548
